# Edge-AI Training — Colab + Local

End-to-end training + INT8 PTQ + FPGA artifact emission. Same `edge_train` Python package backs both this notebook and `train.py` CLI — no logic duplication.

| Section | Purpose |
|---------|---------|
| **0. Environment**  | Detect Colab vs local, install deps, clone repo |
| **1. Config**       | Pick model + dataset + hyperparameters |
| **2. Dataset**      | Load (auto-download), visualize samples |
| **3. Model**        | Build Keras model, print summary |
| **4. Train**        | `fit()` + plot history |
| **5. PTQ INT8**     | TFLite quantize with real calibration set |
| **6. Emit**         | `layer_table.h` + `weights.bin` + `meta.json` |
| **7. Distribute**   | Zip + download (Colab) / paths to deploy.sh (local) |

**Output artifacts** (all consumed by `host/scripts/deploy.sh` -> KV260):
- `firmware/layer_table.h`   — re-build firmware after this changes (`make -C firmware`)
- `training/export/<model>_<dataset>.weights.bin`
- `training/export/<model>_<dataset>_int8.bin.meta.json`


---
# 0. Environment setup


## 0.1 Detect runtime + install deps

Auto-detects Google Colab vs local Jupyter. On Colab we additionally install `kagglehub` and clone the project repo. On local we assume the venv from `training/requirements.txt` is active.

**Colab prerequisite for `cats_dogs` dataset**: upload your `kaggle.json` (Kaggle API token) to `/root/.config/kaggle/`. Get it from your Kaggle account → API → Create New Token. Skip if you'll use `inria` instead (auto-downloads via tarball).


In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Runtime: {'Colab' if IN_COLAB else 'local'}  |  Python {sys.version.split()[0]}")

if IN_COLAB:
    # Replace with your fork or push your project to GitHub
    REPO_URL = "https://github.com/ngtan369/capstoneProject.git"
    REPO_DIR = Path("/content/capstoneProject")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(REPO_DIR)], check=True)
    %cd $REPO_DIR/training
    !pip -q install kagglehub matplotlib
else:
    # Local: assume notebook ran from training/ — adjust if it didn't
    HERE = Path.cwd()
    if HERE.name != "training":
        HERE = next(p for p in [HERE, *HERE.parents] if (p / "edge_train").is_dir())
    %cd $HERE

# Make `import edge_train` resolvable
sys.path.insert(0, str(Path.cwd()))
print(f"cwd: {Path.cwd()}")


## 0.2 Sanity check

Confirm `edge_train` imports cleanly. Failure here usually means cwd is wrong (notebook started outside `training/`) — re-run cell 0.1.


In [ ]:
import edge_train
print(f"edge_train loaded from: {edge_train.__file__}")
print(f"Project root          : {edge_train.project_root()}")
print(f"Firmware dir          : {edge_train.firmware_dir()}")
print(f"Export dir            : {edge_train.export_dir()}")


---
# 1. Configuration


## 1.1 Hyperparameters

`MODEL` — only `vgg-tiny` deploys to FPGA today. Other choices (`vgg11`, `resnet18`, ...) train OK but **export to layer_table.h will fail** (unsupported ops: 1x1, depthwise, BN, residual). Useful for accuracy comparison only.

`DATASET` — `cats_dogs` (Kaggle, ~543 MB) or `inria` (Person, ~969 MB tarball).

`EPOCHS` — VGG-tiny converges quickly (~5–10 epochs at 128×128). On Colab T4 GPU this takes ~2 min/epoch for cats_dogs.


In [ ]:
MODEL      = "vgg-tiny"      # only fully-supported FPGA target
DATASET    = "cats_dogs"     # "cats_dogs" or "inria"
EPOCHS     = 8
BATCH_SIZE = 32

# Optional: skip training and emit artifacts from random init weights — useful
# for hardware bring-up while the dataset / training pipeline is still in flux.
SKIP_TRAINING = False


---
# 2. Dataset


## 2.1 Load + auto-download


In [ ]:
from edge_train import load_dataset, LABEL_MAPS, FPGA_INPUT_SIZE

train_ds, val_ds, num_classes = load_dataset(DATASET, batch_size=BATCH_SIZE)
labels = LABEL_MAPS[DATASET]
print(f"\nClasses: {labels}  (num_classes={num_classes})")
print(f"Image size = {FPGA_INPUT_SIZE} (HxW)  matches FPGA accelerator input")


## 2.2 Visualize random samples

Quick eyeball — verify class labels match folder structure (cats=0, dogs=1).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

batch_x, batch_y = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, lbl in zip(axes.flat, batch_x.numpy()[:8], batch_y.numpy()[:8]):
    ax.imshow(img); ax.set_title(labels[int(lbl)]); ax.axis("off")
plt.tight_layout(); plt.show()


---
# 3. Model


In [ ]:
from edge_train import build_model
model = build_model(MODEL, num_classes)
model.summary()


---
# 4. Train


## 4.1 Compile + fit

Adam optimizer, sparse categorical cross-entropy. Early stopping on val_loss with patience=3 to avoid overfit. Skips entirely if `SKIP_TRAINING = True` (cell 1.1).


In [ ]:
import tensorflow as tf

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

if SKIP_TRAINING:
    print("[!] SKIP_TRAINING=True — using random init weights")
    history = None
else:
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                         restore_best_weights=True),
    ]
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks)


## 4.2 Plot training history


In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["loss"],     label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history.history["accuracy"],     label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()
    print(f"Final val accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")
else:
    print("(no history)")


---
# 5. PTQ INT8 quantization

TFLite quantizes weights AND activations to INT8. Crucial: pass a **real** representative dataset (not random) — `representative_samples()` pulls 100 single-image batches from `val_ds`. Random calibration usually loses 5–10% top-1 accuracy.


In [ ]:
from edge_train import representative_samples, convert_to_tflite_int8

rep_gen = lambda: representative_samples(val_ds, n=100)
tflite_bytes = convert_to_tflite_int8(model, representative_dataset=rep_gen)
print(f"TFLite INT8 size: {len(tflite_bytes)} B ({len(tflite_bytes)/1024:.1f} KB)")


## 5.1 Sanity-check INT8 accuracy

Compare quantized model vs Keras float on a small slice of val_ds. Drop > 3% suggests calibration set is too small or unrepresentative.


In [ ]:
def tflite_eval(tflite_bytes, dataset, n_batches=20):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    in_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
    correct, total = 0, 0
    for i, (xb, yb) in enumerate(dataset):
        if i >= n_batches: break
        for x, y in zip(xb.numpy(), yb.numpy()):
            q = np.round(x / in_d["quantization"][0]) + in_d["quantization"][1]
            q = np.clip(q, -128, 127).astype(np.int8)
            interp.set_tensor(in_d["index"], q[None, ...])
            interp.invoke()
            out = interp.get_tensor(out_d["index"])
            correct += int(out.argmax(axis=-1)[0] == int(y))
            total += 1
    return correct / total if total else 0.0

if not SKIP_TRAINING:
    int8_acc = tflite_eval(tflite_bytes, val_ds, n_batches=20)
    print(f"INT8 TFLite val acc (subset): {int8_acc*100:.2f}%")
else:
    print("(skipped — random weights)")


---
# 6. Emit FPGA artifacts

Parse the TFLite graph -> pack INT8 weights + INT32 biases into `weights.bin`, generate the C header `firmware/layer_table.h`, write `meta.json` for the ARM host.


In [ ]:
from edge_train import emit_layer_table, export_dir, firmware_dir
import json

base = f"{MODEL}_{DATASET}"
out_dir   = export_dir()
out_dir.mkdir(parents=True, exist_ok=True)
blob_path = out_dir / f"{base}.weights.bin"
header_path = firmware_dir() / "layer_table.h"
tflite_path = out_dir / f"{base}_int8.bin"

# Save the TFLite reference (host can use it for a software ground-truth check)
tflite_path.write_bytes(tflite_bytes)

emit_summary = emit_layer_table(tflite_bytes, header_path, blob_path)


## 6.1 Write meta.json + summary


In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
in_d  = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]

meta = {
    "model": MODEL, "dataset": DATASET,
    "dataset_id": edge_train.DATASET_IDS[DATASET],
    "labels":     edge_train.LABEL_MAPS[DATASET],
    "input": {"shape": list(in_d["shape"]), "dtype": str(np.dtype(in_d["dtype"])),
              "scale": float(in_d["quantization"][0]),
              "zero_point": int(in_d["quantization"][1]),
              "fpga_size": list(FPGA_INPUT_SIZE)},
    "output": {"shape": list(out_d["shape"]), "dtype": str(np.dtype(out_d["dtype"])),
               "scale": float(out_d["quantization"][0]),
               "zero_point": int(out_d["quantization"][1])},
    "fpga": emit_summary,
    "weights_file": blob_path.name,
    "header_path":  str(header_path),
}
meta_path = out_dir / f"{base}_int8.bin.meta.json"
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print("Artifacts:")
for p in (header_path, blob_path, meta_path, tflite_path):
    print(f"  {p}  ({p.stat().st_size} B)")
print(f"\nFinal OFM goes to ping-pong buf_idx={emit_summary['final_ofm_buf_idx']}")
print(f"Max tensor bytes (host ping-pong size): {emit_summary['max_tensor_bytes']}")


---
# 7. Distribute artifacts to KV260

**Local**: just run `./host/scripts/deploy.sh` from project root — it rsyncs `firmware/`, `training/export/`, `host/`, and `hw/artifacts/` to the board.

**Colab**: zip the artifacts and download — then on your machine, drop them into the project tree and run `deploy.sh`.


In [ ]:
if IN_COLAB:
    import zipfile
    bundle = Path("/content/edgeai_artifacts.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in (header_path, blob_path, meta_path, tflite_path):
            zf.write(p, arcname=str(p.relative_to(edge_train.project_root())))
    print(f"Zipped: {bundle} ({bundle.stat().st_size/1024:.1f} KB)")
    from google.colab import files
    files.download(str(bundle))
else:
    print("Local run. Next steps:")
    print("  1. Rebuild firmware:   make -C firmware")
    print("  2. Deploy to board:    ./host/scripts/deploy.sh")
    print("  3. SSH + run notebook: ssh ubuntu@kv260.local 'cd ~/edgeai/host && jupyter notebook'")
